# ARGUS — RF-DETR Fine-tune (BDD100K + IDD)

**Model:** RF-DETR Base (COCO AP 53.3)  
**Data:** BDD100K 20K subset + IDD Detection (optional)  
**Classes:** car · motorcycle · bus · truck  
**Est. time:** ~8–10 hrs on T4

### Before running
1. Settings → Accelerator → **GPU T4**
2. Settings → Internet → **On**
3. Add dataset: `a7madmostafa/bdd100k-yolo` (public)
4. Add dataset: `Blank_0013/idd-detection` (your private dataset, optional)
5. **Save Version → Save & Run All (Commit)**

In [ ]:
# ── Cell 1: Install RF-DETR + verify GPU ──────────────────────────────────────
import subprocess
print('Installing rfdetr[train] ...')
subprocess.run(['pip', 'install', '-q', 'rfdetr[train]'], check=True)

import torch
assert torch.cuda.is_available(), 'GPU not found — enable GPU in Settings'
print(f'Torch : {torch.__version__}')
print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
from pathlib import Path

# Dataset inputs — attach these to the notebook
BDD_ROOT = Path('/kaggle/input/bdd100k-yolo')          # a7madmostafa/bdd100k-yolo
IDD_ROOT = Path('/kaggle/input/idd-detection/IDD_Detection')  # optional

# Output
DATASET_DIR = Path('/kaggle/working/argus_dataset')
MODEL_DIR   = Path('/kaggle/working/models')

# Training
EPOCHS      = 30
BATCH_SIZE  = 8     # T4 15.6 GB, RF-DETR base @ 560px — safe
NUM_WORKERS = 4
BDD_TRAIN_N = 20_000  # subset of 70k for speed
BDD_VAL_N   =  2_000

# Classes (4-class ARGUS scheme)
CLASS_NAMES = ['car', 'motorcycle', 'bus', 'truck']

# BDD100K 10-class → ARGUS 4-class
# BDD: 0=person 1=rider 2=car 3=truck 4=bus 5=train 6=motorcycle 7=bicycle 8=light 9=sign
BDD_TO_ARGUS = {2: 0, 6: 1, 4: 2, 3: 3}

# IDD Pascal VOC label strings → ARGUS 4-class
IDD_CLASS_MAP = {
    'car': 0, 'taxi': 0, 'van': 0, 'jeep': 0,
    'motorcycle': 1, 'scooter': 1, 'moped': 1,
    'bus': 2, 'minibus': 2,
    'truck': 3, 'pickup': 3, 'trailer': 3, 'tipper': 3,
    'autorickshaw': 0, 'auto-rickshaw': 0, 'e-rickshaw': 0,
}

USE_IDD = IDD_ROOT.exists()
print(f'BDD100K root  : {BDD_ROOT}  exists={BDD_ROOT.exists()}')
print(f'IDD root      : {IDD_ROOT}  exists={USE_IDD}')
print(f'Classes       : {CLASS_NAMES}')
print(f'Epochs        : {EPOCHS}  batch={BATCH_SIZE}')

In [ ]:
# ── Cell 3: Verify dataset paths ──────────────────────────────────────────────
assert BDD_ROOT.exists(), f'BDD root not found: {BDD_ROOT}\nAttach dataset: a7madmostafa/bdd100k-yolo'

BDD_TRAIN_IMG = BDD_ROOT / 'train' / 'images'
BDD_TRAIN_LBL = BDD_ROOT / 'train' / 'labels'
BDD_VAL_IMG   = BDD_ROOT / 'val'   / 'images'
BDD_VAL_LBL   = BDD_ROOT / 'val'   / 'labels'

for name, p in [('train/images', BDD_TRAIN_IMG), ('train/labels', BDD_TRAIN_LBL),
                ('val/images',   BDD_VAL_IMG),   ('val/labels',   BDD_VAL_LBL)]:
    status = 'OK' if p.exists() else 'MISSING'
    count  = sum(1 for _ in p.iterdir()) if p.exists() else 0
    print(f'  BDD {name:15s}: {status}  ({count:,} files)')

if USE_IDD:
    IDD_ANN = IDD_ROOT / 'Annotations'
    IDD_IMG = IDD_ROOT / 'JPEGImages'
    print(f'  IDD Annotations  : {"OK" if IDD_ANN.exists() else "MISSING"}')
    print(f'  IDD JPEGImages   : {"OK" if IDD_IMG.exists() else "MISSING"}')
else:
    print('  IDD not attached — training on BDD100K only')

In [ ]:
# ── Cell 4: BDD100K YOLO → ARGUS YOLO (remap class IDs) ──────────────────────
import random
import shutil
from collections import Counter

random.seed(42)

def remap_bdd_split(src_img, src_lbl, dst_img, dst_lbl, max_n):
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    imgs = list(src_img.glob('*.jpg')) + list(src_img.glob('*.png'))
    random.shuffle(imgs)

    kept, skipped = 0, 0
    stats = Counter()

    for img_path in imgs:
        if kept >= max_n:
            break
        lbl_path = src_lbl / (img_path.stem + '.txt')
        if not lbl_path.exists():
            skipped += 1
            continue

        new_lines = []
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if not parts:
                continue
            new_cls = BDD_TO_ARGUS.get(int(parts[0]))
            if new_cls is not None:
                new_lines.append(f'{new_cls} {" ".join(parts[1:])}')
                stats[CLASS_NAMES[new_cls]] += 1

        if not new_lines:
            skipped += 1
            continue

        shutil.copy(img_path, dst_img / img_path.name)
        (dst_lbl / (img_path.stem + '.txt')).write_text('\n'.join(new_lines))
        kept += 1

    print(f'  kept={kept:,}  skipped={skipped:,}  classes={dict(stats)}')
    return kept

print('Converting BDD100K train ...')
n_bdd_train = remap_bdd_split(
    BDD_TRAIN_IMG, BDD_TRAIN_LBL,
    DATASET_DIR / 'train' / 'images', DATASET_DIR / 'train' / 'labels',
    BDD_TRAIN_N
)

print('Converting BDD100K val → valid ...')
n_bdd_val = remap_bdd_split(
    BDD_VAL_IMG, BDD_VAL_LBL,
    DATASET_DIR / 'valid' / 'images', DATASET_DIR / 'valid' / 'labels',
    BDD_VAL_N
)

print(f'\nBDD100K: {n_bdd_train:,} train  {n_bdd_val:,} valid')

In [ ]:
# ── Cell 5: IDD Pascal VOC XML → ARGUS YOLO (skipped if not attached) ─────────
import xml.etree.ElementTree as ET
import cv2

def _xyxy_to_yolo(x1, y1, x2, y2, w, h, cls_id):
    cx = ((x1 + x2) / 2) / w
    cy = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return f'{cls_id} {max(0.001,min(0.999,cx)):.6f} {max(0.001,min(0.999,cy)):.6f} '\
           f'{max(0.001,min(0.999,bw)):.6f} {max(0.001,min(0.999,bh)):.6f}'

def parse_idd_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    iw = int(size.find('width').text)  if size is not None else 0
    ih = int(size.find('height').text) if size is not None else 0
    lines = []
    for obj in root.findall('object'):
        name_el = obj.find('name')
        if name_el is None:
            continue
        cls_id = IDD_CLASS_MAP.get(name_el.text.strip().lower())
        if cls_id is None:
            continue
        bb = obj.find('bndbox')
        if bb is None:
            continue
        x1, y1 = float(bb.find('xmin').text), float(bb.find('ymin').text)
        x2, y2 = float(bb.find('xmax').text), float(bb.find('ymax').text)
        if x2 > x1 and y2 > y1 and iw > 0 and ih > 0:
            lines.append(_xyxy_to_yolo(x1, y1, x2, y2, iw, ih, cls_id))
    return iw, ih, lines

def convert_idd_split(split, out_img, out_lbl):
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    ann_root = IDD_ROOT / 'Annotations' / split
    img_root = IDD_ROOT / 'JPEGImages' / split
    if not ann_root.exists():
        print(f'  IDD {split}: Annotations dir not found — skipping')
        return 0
    written = 0
    for seq_dir in sorted(ann_root.iterdir()):
        if not seq_dir.is_dir():
            continue
        for xml_file in seq_dir.glob('*.xml'):
            stem = xml_file.stem
            iw, ih, lines = parse_idd_xml(xml_file)
            if not lines:
                continue
            img_path = None
            for ext in ('.jpg', '.jpeg', '.png'):
                candidate = img_root / seq_dir.name / (stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            if img_path is None:
                continue
            out_name = f'idd_{seq_dir.name}_{stem}'
            dst_img = out_img / (out_name + '.jpg')
            if img_path.suffix == '.jpg':
                shutil.copy(img_path, dst_img)
            else:
                img = cv2.imread(str(img_path))
                cv2.imwrite(str(dst_img), img, [cv2.IMWRITE_JPEG_QUALITY, 90])
            (out_lbl / (out_name + '.txt')).write_text('\n'.join(lines))
            written += 1
    print(f'  IDD {split}: {written:,} images')
    return written

n_idd_train = n_idd_val = 0
if USE_IDD:
    print('Converting IDD train ...')
    n_idd_train = convert_idd_split('train',
        DATASET_DIR / 'train' / 'images', DATASET_DIR / 'train' / 'labels')
    print('Converting IDD val → valid ...')
    n_idd_val = convert_idd_split('val',
        DATASET_DIR / 'valid' / 'images', DATASET_DIR / 'valid' / 'labels')
else:
    print('IDD not attached — skipping')

In [ ]:
# ── Cell 6: Dataset summary + data.yaml ───────────────────────────────────────
train_img_dir = DATASET_DIR / 'train' / 'images'
valid_img_dir = DATASET_DIR / 'valid' / 'images'

n_train = sum(1 for _ in train_img_dir.iterdir())
n_valid = sum(1 for _ in valid_img_dir.iterdir())

print('── Dataset summary ─────────────────────────────')
print(f'  BDD100K  train={n_bdd_train:,}  valid={n_bdd_val:,}')
print(f'  IDD      train={n_idd_train:,}  valid={n_idd_val:,}')
print(f'  TOTAL    train={n_train:,}  valid={n_valid:,}')
print(f'  Classes  : {CLASS_NAMES}')

# RF-DETR YOLO loader requires the split dir to be named 'valid', not 'val'
yaml_content = f"""path: {DATASET_DIR.resolve()}
train: train/images
val:   valid/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
yaml_path = DATASET_DIR / 'data.yaml'
yaml_path.write_text(yaml_content)
print(f'\ndata.yaml → {yaml_path}')
print(yaml_content)

In [ ]:
# ── Cell 7: Train RF-DETR ─────────────────────────────────────────────────────
import os
from rfdetr import RFDETRBase

RUN_DIR = MODEL_DIR / 'rfdetr_run'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'Training RF-DETR for {EPOCHS} epochs ...')
print(f'  dataset : {DATASET_DIR}')
print(f'  output  : {RUN_DIR}')
print(f'  batch   : {BATCH_SIZE}  workers={NUM_WORKERS}')

model = RFDETRBase()
model.train(
    dataset_file           = 'yolo',
    dataset_dir            = str(DATASET_DIR.resolve()),
    output_dir             = str(RUN_DIR),
    epochs                 = EPOCHS,
    batch_size             = BATCH_SIZE,
    lr                     = 5e-5,
    lr_encoder             = 7.5e-5,
    device                 = 'cuda',
    class_names            = CLASS_NAMES,
    progress_bar           = 'tqdm',
    tensorboard            = False,
    checkpoint_interval    = 5,
    early_stopping         = True,
    early_stopping_patience= 8,
    num_workers            = NUM_WORKERS,
    multi_scale            = True,
    expanded_scales        = True,
)

In [ ]:
# ── Cell 8: Save checkpoint + instructions ────────────────────────────────────
import shutil

FINAL = Path('/kaggle/working/argus_rfdetr_finetuned.pth')

for candidate in ['checkpoint_best_total.pth', 'checkpoint_best_ap50.pth', 'checkpoint_last.pth']:
    src = RUN_DIR / candidate
    if src.exists():
        shutil.copy(src, FINAL)
        print(f'Saved: {FINAL}  (from {candidate})')
        print(f'Size : {FINAL.stat().st_size / 1e6:.1f} MB')
        break
else:
    print('WARNING: no checkpoint found in', RUN_DIR)
    for f in sorted(RUN_DIR.iterdir()):
        print(' ', f.name)

print()
print('── Next steps ───────────────────────────────────')
print('1. Download argus_rfdetr_finetuned.pth from Output tab')
print('2. Place at ARGUS/rf-detr-base.pth  (replaces COCO weights)')
print('   OR pass via VehicleDetector(pretrain_weights="path/to/argus_rfdetr_finetuned.pth")')
print('3. Run week3_retrain.py to fine-tune further on ARGUS dashcam data')